## 准备数据

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [3]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [4]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.truncated_normal([28 * 28, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.truncated_normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 28 * 28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [5]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [11]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.90161175 ; accuracy 0.7938667
epoch 1 : loss 0.8994729 ; accuracy 0.7943
epoch 2 : loss 0.89734805 ; accuracy 0.79461664
epoch 3 : loss 0.895237 ; accuracy 0.79501665
epoch 4 : loss 0.89313984 ; accuracy 0.79541665
epoch 5 : loss 0.891056 ; accuracy 0.7959167
epoch 6 : loss 0.8889857 ; accuracy 0.7964
epoch 7 : loss 0.8869288 ; accuracy 0.79681665
epoch 8 : loss 0.8848849 ; accuracy 0.79716665
epoch 9 : loss 0.8828542 ; accuracy 0.79753333
epoch 10 : loss 0.8808365 ; accuracy 0.79803336
epoch 11 : loss 0.87883174 ; accuracy 0.7984833
epoch 12 : loss 0.8768398 ; accuracy 0.79901665
epoch 13 : loss 0.8748605 ; accuracy 0.79941666
epoch 14 : loss 0.8728938 ; accuracy 0.79966664
epoch 15 : loss 0.8709393 ; accuracy 0.80011666
epoch 16 : loss 0.86899716 ; accuracy 0.80053335
epoch 17 : loss 0.86706734 ; accuracy 0.80085
epoch 18 : loss 0.8651492 ; accuracy 0.8013333
epoch 19 : loss 0.8632432 ; accuracy 0.80175
epoch 20 : loss 0.8613491 ; accuracy 0.8021167
epoch 21 : loss 0